# 1244. Design A Leaderboard

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** design, hash-table, sorting, heap
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/design-a-leaderboard/)

Design a Leaderboard class, which has three functions:

- `addScore(playerId, score)` - update the leaderboard by adding `score` to the given
  player's score. If there is no player with such id in the leaderboard, add them with
  the given `score`.
- `top(K)` - return the score sum of the top `K` players.
- `reset(playerId)` - reset the score of the player with the given id to `0` (in other
  words, erase it from the leaderboard). It is guaranteed that the player was added to
  the leaderboard before calling this function.

Initially, the leaderboard is empty.

---

### Example

```
Leaderboard l = new Leaderboard();
l.addScore(1, 73);   // leaderboard = [[1,73]]
l.addScore(2, 56);   // leaderboard = [[1,73],[2,56]]
l.addScore(3, 39);   // leaderboard = [[1,73],[2,56],[3,39]]
l.addScore(4, 51);   // leaderboard = [[1,73],[2,56],[3,39],[4,51]]
l.addScore(5, 4);    // leaderboard = [[1,73],[2,56],[3,39],[4,51],[5,4]]
l.top(1);            // 73
l.reset(1);          // leaderboard = [[2,56],[3,39],[4,51],[5,4]]
l.reset(2);          // leaderboard = [[3,39],[4,51],[5,4]]
l.addScore(2, 51);   // leaderboard = [[2,51],[3,39],[4,51],[5,4]]
l.top(3);            // 141 = 51 + 51 + 39
```

---

### Constraints

- `1 <= playerId, K <= 10000`
- It is guaranteed that `K` is less than or equal to the current number of players
- `1 <= score <= 100`
- At most `1000` function calls will be made

Every game you have played has this class in it. LeetCode's limits are small enough
that the lazy answer is the right answer - which makes this the best problem in the
repo for practising the sentence "I would sort here, and here is the number of players
at which I would stop."

## Before you write anything

**1.** `addScore` has to do two different things - create a player, and add to an
existing one. Write it as an `if`/`else`, then write it as **one line** with a dict
method that makes the two cases the same. (You have used the same trick in #359 and
#242; the method's name is in both.)

**2.** `reset` says "reset to `0` (in other words, erase it)". Those are two different
implementations: set the value to `0`, or delete the key. Does the difference show up
in `top(K)`? Construct a case where it would matter, then check it against the
constraint that `K` is at most the current number of players - and decide whether the
constraint saves you or whether you are relying on luck.

**3.** `top(K)` needs the sum of the `K` biggest values. Cost three implementations for
`n` players:

```
sort all the scores, take K          ->  O(?)
heapq.nlargest(K, scores)            ->  O(?)
keep a permanently sorted structure  ->  O(?) per addScore, O(?) per top
```

At most `1000` calls, so `n <= 1000`. Compute the actual number of operations for the
first two at `n = 1000`, `K = 10`, and then say - honestly - whether the difference is
worth one extra line of code *here*.

**4.** Now change the problem: a real leaderboard has ten million players and `top(10)`
is called on every page load. Which of your three answers survives, and what does the
third one have to be? Write down what breaks about "sort on every query" at that size -
in seconds, not in big-O.

**5.** Scores only ever go **up** (`1 <= score`) until a `reset`. Does that help? Think
about what it would let you cache, and what `reset` would have to invalidate. (This is
the kind of question whose answer is sometimes "no" - say so if so, and say why.)

**6.** How would you test it? Only `top` returns anything, so a `reset` that fails to
erase, or an `addScore` that overwrites instead of adding, is invisible until a `top`
call happens to include that player. What would you check after every call?

## Two routes

**A - one dict, sort on demand** *(write this first)*

```
self.scores = {}                       # playerId -> total score
```

- `addScore`: `self.scores[playerId] = self.scores.get(playerId, 0) + score`. One
  line, and the "new player" case disappears into the default.
- `top(K)`: `sum(sorted(self.scores.values(), reverse=True)[:K])`.
- `reset`: `self.scores.pop(playerId, None)`.

`addScore` and `reset` are `O(1)`; `top` is `O(n log n)`. With `n <= 1000` and at most
1000 calls that is at most about ten million operations in the worst case - fine, and
this is the accepted answer.

Swapping `sorted(...)[:K]` for `heapq.nlargest(K, ...)` takes it to `O(n log K)`, which
at `K = 10` is a real improvement for one character more of thinking. Do it, but be
able to say that it was not *necessary* here.

**B - keep the scores bucketed** *(question 4)*

Scores are bounded: `1 <= score <= 100` and at most 1000 calls, so no total exceeds
100 000. Keep `counts[score] = how many players have that score` and walk **down** from
the top, taking players until you have `K`. `top(K)` becomes `O(max_score)` -
independent of the number of players - and `addScore` stays `O(1)` by decrementing the
old bucket and incrementing the new one.

That is the shape real leaderboards use, and it is why the score range being bounded is
in the constraints at all. When the range is *not* bounded you reach for a sorted
container - a skip list or a balanced tree - which is exactly what Redis's sorted set
is, and why every game backend is built on one.

> **The right answer depends on `n`, and `n` is in the constraints.** Route A is correct
> and shippable *at this size*. The habit worth building is not "always use a heap" - it
> is reading the constraints, doing the arithmetic, and being able to name the size at
> which your answer stops working.

In [17]:
class Node:
    def __init__(self, data, color="red", left=None, right=None, parent=None):
        self.data = data
        self.color = color
        self.left = left
        self.right = right
        self.parent = parent



class RedBlackTree:
    def __init__(self):
        self.NIL = Node(data=[], color="black")  # Sentinel NIL node
        self.root = self.NIL
        self.by_id = {}  # playerId -> Node, for O(1) lookup by id

    def rotate_left(self, x):
        y = x.right
        x.right = y.left
        if y.left != self.NIL:
            y.left.parent = x
        y.parent = x.parent
        if x.parent is None:
            self.root = y
        elif x == x.parent.left:
            x.parent.left = y
        else:
            x.parent.right = y
        y.left = x
        x.parent = y

    def rotate_right(self, y):
        x = y.left
        y.left = x.right
        if x.right != self.NIL:
            x.right.parent = y
        x.parent = y.parent
        if y.parent is None:
            self.root = x
        elif y == y.parent.right:
            y.parent.right = x
        else:
            y.parent.left = x
        x.right = y
        y.parent = x

    def insert(self, data):
        new_node = Node(data=data)
        new_node.left = new_node.right = self.NIL
        self.by_id[new_node.data[0]] = new_node

        parent = None
        current = self.root

        while current != self.NIL:
            parent = current
            if new_node.data[1] < current.data[1]:
                current = current.left
            else:
                current = current.right

        new_node.parent = parent
        if parent is None:
            self.root = new_node
        elif new_node.data[1] < parent.data[1]:
            parent.left = new_node
        else:
            parent.right = new_node

        self.fix_insert(new_node)

    def fix_insert(self, z):
        # Case: Parent is red, needing adjustment
        while z.parent and z.parent.color == "red":
            if z.parent == z.parent.parent.left:
                y = z.parent.parent.right  # Uncle node
                if y.color == "red":
                    # Case 2: Both parent and uncle are red
                    z.parent.color = "black"
                    y.color = "black"
                    z.parent.parent.color = "red"
                    z = z.parent.parent  # Recurse upward
                else:
                    # Case 3: Parent is red, uncle is black, and z is a right child
                    if z == z.parent.right:
                        z = z.parent
                        self.rotate_left(z)  # Left-rotate to correct shape
                    # Case 3: Left-rotation done, recolor and rotate
                    z.parent.color = "black"
                    z.parent.parent.color = "red"
                    self.rotate_right(z.parent.parent)  # Right-rotate to fix violation
            else:
                # Symmetric cases for when z's parent is the right child
                y = z.parent.parent.left
                if y.color == "red":
                    # Case 2: Parent and uncle are both red
                    z.parent.color = "black"
                    y.color = "black"
                    z.parent.parent.color = "red"
                    z = z.parent.parent
                else:
                    # Case 3: Parent is red, uncle is black, and z is a left child
                    if z == z.parent.left:
                        z = z.parent
                        self.rotate_right(z)
                    # Case 3: Recoloring and left-rotation
                    z.parent.color = "black"
                    z.parent.parent.color = "red"
                    self.rotate_left(z.parent.parent)
        # Case 1: Root is always black after insertion fix
        self.root.color = "black"

    def transplant(self, u, v):
        if u.parent is None:
            self.root = v
        elif u == u.parent.left:
            u.parent.left = v
        else:
            u.parent.right = v
        v.parent = u.parent

    def delete(self, id):
        z = self.search_by_id(id)
        if z == self.NIL:
            print("Value not found in the tree.")
            return
        del self.by_id[id]

        y = z
        y_original_color = y.color
        if z.left == self.NIL:
            x = z.right
            self.transplant(z, z.right)
        elif z.right == self.NIL:
            x = z.left
            self.transplant(z, z.left)
        else:
            y = self.minimum(z.right)
            y_original_color = y.color
            x = y.right
            if y.parent == z:
                x.parent = y
            else:
                self.transplant(y, y.right)
                y.right = z.right
                y.right.parent = y
            self.transplant(z, y)
            y.left = z.left
            y.left.parent = y
            y.color = z.color

        if y_original_color == "black":
            self.fix_delete(x)

    def fix_delete(self, x):
        while x != self.root and x.color == "black":
            if x == x.parent.left:
                w = x.parent.right  # Sibling node
                if w.color == "red":
                    # Case 1: Sibling is red
                    w.color = "black"
                    x.parent.color = "red"
                    self.rotate_left(x.parent)
                    w = x.parent.right
                if w.left.color == "black" and w.right.color == "black":
                    # Case 2: Sibling and its children are black
                    w.color = "red"
                    x = x.parent  # Move up the tree
                else:
                    if w.right.color == "black":
                        # Case 3: Sibling is black, left child is red, right is black
                        w.left.color = "black"
                        w.color = "red"
                        self.rotate_right(w)
                        w = x.parent.right
                    # Case 3: Right child of sibling is red
                    w.color = x.parent.color
                    x.parent.color = "black"
                    w.right.color = "black"
                    self.rotate_left(x.parent)
                    x = self.root
            else:
                # Symmetric cases for when x is the right child
                w = x.parent.left
                if w.color == "red":
                    # Case 1: Sibling is red
                    w.color = "black"
                    x.parent.color = "red"
                    self.rotate_right(x.parent)
                    w = x.parent.left
                if w.right.color == "black" and w.left.color == "black":
                    # Case 2: Sibling and its children are black
                    w.color = "red"
                    x = x.parent
                else:
                    if w.left.color == "black":
                        # Case 3: Sibling is black, right child is red, left is black
                        w.right.color = "black"
                        w.color = "red"
                        self.rotate_left(w)
                        w = x.parent.left
                    # Case 3: Left child of sibling is red
                    w.color = x.parent.color
                    x.parent.color = "black"
                    w.left.color = "black"
                    self.rotate_right(x.parent)
                    x = self.root
        # Ensure the final node is black
        x.color = "black"

    def search_by_id(self, id):
        return self.by_id.get(id, self.NIL)

    def minimum(self, node):
        while node.left != self.NIL:
            node = node.left
        return node

    def inorder(self , l):
        lst = []
        self._inorderRev(self.root, lst , l)
        return lst

    def _inorderRev(self, node, lst: list , l):
        # Right-to-left traversal: highest score first
        if node != self.NIL and  len(lst) <l  :
            self._inorderRev(node.right, lst ,l)
            if len(lst) <l :
                lst.append(node.data)
                self._inorderRev(node.left, lst ,l)


In [15]:
class Leaderboard:

    def __init__(self):
        self.Tree:RedBlackTree = RedBlackTree()

    def addScore(self, playerId: int, score: int) -> None:
        i = self.Tree.by_id.get(playerId)
        if i :
            score += i.data[1]
            self.Tree.delete(playerId)
        self.Tree.insert([playerId,score])

    def top(self, K: int) -> int:
        return sum(score for _, score in self.Tree.inorder(K))

    def reset(self, playerId: int) -> None:
        self.Tree.delete(playerId)

### The test harness

Question 6's answer. `addScore` and `reset` return nothing, so an `addScore` that
**overwrites** instead of accumulating, or a `reset` that leaves the player in place,
is silent until a `top` call happens to reach that player's row.

So `check` replays a call sequence against your class **and** against a plain dict
model. After **every** call it asks your leaderboard for `top(K)` at *every* legal `K`
from `1` to the number of players, and compares each one against the model. Probing
every `K` rather than just the one the test asked for is what turns "player 3's score
is wrong" into a failure at the call that caused it - a wrong score in the middle of
the ranking is invisible at `top(1)` and obvious at `top(3)`.

`stress` uses a small pool of players so ids are re-used constantly and scores
accumulate, with resets mixed in. Run this cell; don't edit it.

In [7]:
import random


def check(ops):
    '''Replay (op, args) against Leaderboard and a dict model, probing every K after each call.'''
    log = []
    try:
        lb = Leaderboard()
    except Exception as e:
        return False, [f"   !! Leaderboard() raised {type(e).__name__}: {e}"]

    model = {}

    def want(k):
        return sum(sorted(model.values(), reverse=True)[:k])

    for op, args in ops:
        call = f"{op}({', '.join(map(str, args))})"
        try:
            if op == "addScore":
                p, s = args
                lb.addScore(p, s)
                model[p] = model.get(p, 0) + s
                log.append(f"{call}        scores {dict(sorted(model.items()))}")
            elif op == "reset":
                p = args[0]
                lb.reset(p)
                model.pop(p, None)
                log.append(f"{call}            scores {dict(sorted(model.items()))}")
            else:
                k = args[0]
                got = lb.top(k)
                log.append(f"{call} -> {got!r}   (want {want(k)})")
                if got != want(k):
                    log.append(f"   !! {call} must return {want(k)}, got {got!r}")
                    log.append(f"      scores are {dict(sorted(model.items()))}")
                    return False, log
        except Exception as e:
            log.append(f"   !! {call} raised {type(e).__name__}: {e}")
            return False, log

        # after EVERY call, probe every legal K
        for k in range(1, len(model) + 1):
            try:
                got = lb.top(k)
            except Exception as e:
                log.append(f"   !! after {call}, top({k}) raised {type(e).__name__}: {e}")
                return False, log
            if got != want(k):
                log.append(f"   !! after {call}, top({k}) is {got!r}, should be {want(k)}")
                log.append(f"      scores are {dict(sorted(model.items()))}")
                log.append(f"      sorted descending: {sorted(model.values(), reverse=True)}")
                return False, log

    return True, log


def stress(n, seed=0, players=6, max_score=20):
    '''A small pool of players, so ids repeat and scores must accumulate.'''
    random.seed(seed)
    ops, live = [], set()
    for _ in range(n):
        r = random.random()
        if r < 0.6 or not live:
            p = random.randint(1, players)
            ops.append(("addScore", (p, random.randint(1, max_score))))
            live.add(p)
        elif r < 0.8:
            p = random.choice(sorted(live))
            ops.append(("reset", (p,)))
            live.discard(p)
        else:
            ops.append(("top", (random.randint(1, len(live)),)))
    return check(ops)


def report(name, ok, log, tail=6):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")

In [18]:
# tests
CASES = [
    ("the LeetCode example", [
        ("addScore", (1, 73)), ("addScore", (2, 56)), ("addScore", (3, 39)),
        ("addScore", (4, 51)), ("addScore", (5, 4)),
        ("top", (1,)), ("reset", (1,)), ("reset", (2,)),
        ("addScore", (2, 51)), ("top", (3,))]),

    ("question 1: addScore ACCUMULATES, it does not overwrite", [
        ("addScore", (1, 10)), ("addScore", (1, 10)), ("addScore", (1, 10)),
        ("top", (1,))]),                                       # 30, not 10

    ("one player only", [("addScore", (7, 5)), ("top", (1,))]),

    ("K equal to the number of players", [
        ("addScore", (1, 1)), ("addScore", (2, 2)), ("addScore", (3, 3)),
        ("top", (3,))]),

    ("ties everywhere", [
        ("addScore", (i, 10)) for i in range(1, 6)] + [("top", (1,)), ("top", (3,)), ("top", (5,))]),

    ("question 2: reset then re-add starts from zero", [
        ("addScore", (1, 100)), ("reset", (1,)), ("addScore", (1, 5)), ("top", (1,))]),

    ("reset a player who is not the top", [
        ("addScore", (1, 90)), ("addScore", (2, 50)), ("addScore", (3, 10)),
        ("reset", (2,)), ("top", (2,))]),                      # 100 = 90 + 10

    ("reset every player, then start again", [
        ("addScore", (1, 5)), ("addScore", (2, 5)),
        ("reset", (1,)), ("reset", (2,)),
        ("addScore", (9, 7)), ("top", (1,))]),

    ("the minimum score is 1", [
        ("addScore", (1, 1)), ("addScore", (2, 1)), ("top", (2,))]),

    ("many players, top of a long tail", [
        ("addScore", (i, i)) for i in range(1, 51)] + [("top", (1,)), ("top", (10,)), ("top", (50,))]),
]

for name, ops in CASES:
    report(name, *check(ops))

for n, seed, players, mx in [(30, 1, 3, 10), (100, 2, 6, 20), (300, 3, 15, 100), (600, 4, 40, 100)]:
    report(f"stress: {n} calls (seed {seed}, {players} players, scores 1..{mx})",
           *stress(n, seed, players, mx))

print("\ntrace of the LeetCode example:")
for line in check(CASES[0][1])[1]:
    print("  " + line)

OK   the LeetCode example
OK   question 1: addScore ACCUMULATES, it does not overwrite
OK   one player only
OK   K equal to the number of players
OK   ties everywhere
OK   question 2: reset then re-add starts from zero
OK   reset a player who is not the top
OK   reset every player, then start again
OK   the minimum score is 1
OK   many players, top of a long tail
OK   stress: 30 calls (seed 1, 3 players, scores 1..10)
OK   stress: 100 calls (seed 2, 6 players, scores 1..20)
OK   stress: 300 calls (seed 3, 15 players, scores 1..100)
OK   stress: 600 calls (seed 4, 40 players, scores 1..100)

trace of the LeetCode example:
  addScore(1, 73)        scores {1: 73}
  addScore(2, 56)        scores {1: 73, 2: 56}
  addScore(3, 39)        scores {1: 73, 2: 56, 3: 39}
  addScore(4, 51)        scores {1: 73, 2: 56, 3: 39, 4: 51}
  addScore(5, 4)        scores {1: 73, 2: 56, 3: 39, 4: 51, 5: 4}
  top(1) -> 73   (want 73)
  reset(1)            scores {2: 56, 3: 39, 4: 51, 5: 4}
  reset(2)         

## After it passes

- **Do question 3's arithmetic for real.** Build 10 000 players, then `timeit`
  `top(10)` with `sorted(...)[:K]` and with `heapq.nlargest(K, ...)`. Write both
  numbers down. Then answer the question that matters: at what number of players does
  the sorting version cross 100 ms, which is roughly where a user notices?
- **Build route B** (the bucket-by-score version) and run the same tests. Then time
  `top(10)` on 10 000 players in all three. The bucket version should be flat as the
  player count grows, which is the property you were buying.
- **The invariant.** *`self.scores` contains exactly the players who have a non-zero
  score, and each value is the sum of every `addScore` for that player since their last
  `reset`.* Name the two lines that could break the second half of that sentence.
- **Make it real.** Three features, each of which breaks a different design:
  `rank(playerId)` - what position is this player in? (route A must sort; the buckets
  can count downwards). `top(K)` returning the **ids** and not just the sum - suddenly
  ties need a rule, so what is it? And weekly leaderboards, where scores expire - now
  every entry needs a timestamp and you are back in #359's territory.
- **Then look up what you have built.** A structure with `O(log n)` insert and
  "give me the top K" is a **sorted set**, and Redis ships one as `ZADD`/`ZREVRANGE`
  precisely because every game and every leaderboard needs it. You have now written the
  slow version, so the fast one will make sense.
- Siblings: **#295 Find Median from Data Stream** (the same "I only need part of the
  order" idea, with heaps), #347 Top K Frequent Elements (top-K without the running
  totals), #1396 Design Underground System (a dict of running totals, without the
  ranking), #155 Min Stack.